<a href="https://colab.research.google.com/github/VICTOR-hub520/crop-yield-app-12/blob/main/Crop_Yield_Prediction_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crop Yield Prediction Analysis

A comprehensive tool for predicting crop yields based on environmental factors using machine learning.

**Features:**
- Single Crop Yield Prediction
- Sensitivity Analysis
- Batch CSV Processing
- Multi-Scenario Trends & Comparison
- Image-based Crop Analysis
- Advanced Crop Advisor Bot

## Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q scikit-learn joblib pandas numpy plotly pillow requests

import numpy as np
import pandas as pd
import joblib
import plotly.graph_objects as go
import plotly.express as px
from io import BytesIO
from PIL import Image
import requests
import re
import time
from urllib.parse import quote

print("All dependencies installed successfully!")

## Load Model & Configuration

In [ ]:
# Download the pre-trained model from GitHub
!wget -q -O crop_yield_model.pkl https://github.com/VICTOR-hub520/crop-yield-app-12/raw/main/crop_yield_model.pkl

# Load the model
try:
    model = joblib.load('crop_yield_model.pkl')
    print("✓ Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

In [ ]:
# Feature names for the model
hardcoded_features = [
    'Year', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp',
    'Area_Algeria', 'Area_Angola', 'Area_Argentina', 'Area_Armenia', 'Area_Australia', 'Area_Austria',
    'Area_Azerbaijan', 'Area_Bahamas', 'Area_Bahrain', 'Area_Bangladesh', 'Area_Belarus', 'Area_Belgium',
    'Area_Botswana', 'Area_Brazil', 'Area_Bulgaria', 'Area_Burkina Faso', 'Area_Burundi', 'Area_Cameroon',
    'Area_Canada', 'Area_Central African Republic', 'Area_Chile', 'Area_Colombia', 'Area_Croatia', 'Area_Denmark',
    'Area_Dominican Republic', 'Area_Ecuador', 'Area_Egypt', 'Area_El Salvador', 'Area_Eritrea', 'Area_Estonia',
    'Area_Finland', 'Area_France', 'Area_Germany', 'Area_Ghana', 'Area_Greece', 'Area_Guatemala', 'Area_Guinea',
    'Area_Guyana', 'Area_Haiti', 'Area_Honduras', 'Area_Hungary', 'Area_India', 'Area_Indonesia', 'Area_Iraq',
    'Area_Ireland', 'Area_Italy', 'Area_Jamaica', 'Area_Japan', 'Area_Kazakhstan', 'Area_Kenya', 'Area_Latvia',
    'Area_Lebanon', 'Area_Lesotho', 'Area_Libya', 'Area_Lithuania', 'Area_Madagascar', 'Area_Malawi', 'Area_Malaysia',
    'Area_Mali', 'Area_Mauritania', 'Area_Mauritius', 'Area_Mexico', 'Area_Montenegro', 'Area_Morocco', 'Area_Mozambique',
    'Area_Namibia', 'Area_Nepal', 'Area_Netherlands', 'Area_New Zealand', 'Area_Nicaragua', 'Area_Niger', 'Area_Norway',
    'Area_Pakistan', 'Area_Papua New Guinea', 'Area_Peru', 'Area_Poland', 'Area_Portugal', 'Area_Qatar', 'Area_Romania',
    'Area_Rwanda', 'Area_Saudi Arabia', 'Area_Senegal', 'Area_Slovenia', 'Area_South Africa', 'Area_Spain', 'Area_Sri Lanka',
    'Area_Sudan', 'Area_Suriname', 'Area_Sweden', 'Area_Switzerland', 'Area_Tajikistan', 'Area_Thailand', 'Area_Tunisia',
    'Area_Turkey', 'Area_Uganda', 'Area_Ukraine', 'Area_United Kingdom', 'Area_Uruguay', 'Area_Zambia', 'Area_Zimbabwe',
    'Item_Maize', 'Item_Plantains and others', 'Item_Potatoes', 'Item_Rice, paddy', 'Item_Sorghum', 'Item_Soybeans',
    'Item_Sweet potatoes', 'Item_Wheat', 'Item_Yams'
]

# Use model's feature names if available
all_features = list(model.feature_names_in_) if hasattr(model, 'feature_names_in_') else hardcoded_features

# Extract available areas and items
areas = sorted([f.replace("Area_", "") for f in all_features if f.startswith("Area_")])
items = sorted([f.replace("Item_", "") for f in all_features if f.startswith("Item_")])

print(f"Available Countries: {len(areas)}")
print(f"Available Crops: {len(items)}")
print(f"Total Features: {len(all_features)}")

## Helper Functions

In [ ]:
def predict_yield(rainfall, temperature, pesticides, year, area, item):
    """Predict crop yield for given parameters."""
    try:
        feature_dict = {name: 0.0 for name in all_features}
        feature_dict['Year'] = year
        feature_dict['average_rain_fall_mm_per_year'] = rainfall
        feature_dict['pesticides_tonnes'] = pesticides
        feature_dict['avg_temp'] = temperature
        feature_dict[f'Area_{area}'] = 1.0
        feature_dict[f'Item_{item}'] = 1.0
        
        features = pd.DataFrame([feature_dict])
        prediction = model.predict(features)[0]
        return prediction
    except Exception as e:
        print(f"Prediction error: {e}")
        return None

def hectares_to_acres(hectares):
    """Convert hectares to acres."""
    return hectares * 2.471

def hg_per_hectare_to_lbs_per_acre(hg_ha):
    """Convert hg/ha to lbs/acre."""
    return (hg_ha * 0.0220462) / 2.471

def hg_per_hectare_to_tons_per_acre(hg_ha):
    """Convert hg/ha to short tons/acre."""
    lbs_per_acre = hg_per_hectare_to_lbs_per_acre(hg_ha)
    return lbs_per_acre / 2000

print("Helper functions loaded successfully!")

## Crop Advisor Knowledge Base

In [ ]:
advisor_crops = items + [
    "Barley", "Cotton", "Tomatoes", "Sugarcane", "Coffee", "Cocoa",
    "Bananas", "Peanuts", "Sunflower", "Cassava", "Onion", "Pepper"
]

issue_keywords = {
    "drought": "Water Stress",
    "dry": "Water Stress",
    "wilting": "Water Stress",
    "yellow": "Nutrient Deficiency",
    "yellowing": "Nutrient Deficiency",
    "stunted": "Nutrient Deficiency",
    "pest": "Pests/Diseases",
    "disease": "Pests/Diseases",
    "spots": "Pests/Diseases",
    "blight": "Pests/Diseases",
    "rot": "Pests/Diseases",
    "weed": "Weed Pressure",
    "heat": "Weather Stress",
    "cold": "Weather Stress",
    "frost": "Weather Stress"
}

crop_advice = {
    "Wheat": {
        "Pests/Diseases": "Implement crop rotation, use resistant varieties, and scout regularly for rust and aphids.",
        "Water Stress": "Maintain even soil moisture during tillering and heading. Irrigate during dry spells.",
        "Nutrient Deficiency": "Apply balanced NPK fertilizer. Test soil and address specific deficiencies.",
        "Weather Stress": "Protect from late frost with appropriate planting dates."
    },
    "Maize": {
        "Pests/Diseases": "Scout for fall armyworm and borers. Use biological controls when possible.",
        "Water Stress": "Ensure consistent moisture from silking to grain fill.",
        "Nutrient Deficiency": "Maize needs high nitrogen and phosphorus. Split applications recommended.",
    },
    "Rice, paddy": {
        "Pests/Diseases": "Practice proper water management and use resistant varieties.",
        "Water Stress": "Maintain consistent water depth before heading.",
        "Soil Health": "Maintain pH near 6-7 and add organic matter."
    }
}

print("Crop advisor knowledge loaded!")

# SECTION 1: Single Crop Prediction

In [ ]:
print("=" * 60)
print("CROP YIELD PREDICTION - SINGLE SCENARIO")
print("=" * 60)

# Parameters
prediction_year = 2025
rainfall_mm = 1200
temperature_c = 25
pesticides_tonnes = 10
country = "India"
crop = "Wheat"

print(f"\nInput Parameters:")
print(f"  Year: {prediction_year}")
print(f"  Rainfall: {rainfall_mm} mm/year")
print(f"  Temperature: {temperature_c}°C")
print(f"  Pesticides: {pesticides_tonnes} tonnes")
print(f"  Country: {country}")
print(f"  Crop: {crop}")

# Make prediction
yield_hg_ha = predict_yield(rainfall_mm, temperature_c, pesticides_tonnes, prediction_year, country, crop)

if yield_hg_ha:
    yield_lbs_acre = hg_per_hectare_to_lbs_per_acre(yield_hg_ha)
    yield_tons_acre = hg_per_hectare_to_tons_per_acre(yield_hg_ha)
    
    print(f"\nPredicted Yield:")
    print(f"  {yield_hg_ha:,.0f} hg/ha (hectograms per hectare)")
    print(f"  {yield_lbs_acre:,.0f} lbs/acre (pounds per acre)")
    print(f"  {yield_tons_acre:,.4f} tons/acre (short tons per acre)")
else:
    print("Error in prediction.")

# SECTION 2: Sensitivity Analysis

In [ ]:
print("\n" + "=" * 60)
print("SENSITIVITY ANALYSIS")
print("=" * 60)

# Base parameters
base_rainfall = 1000
base_temp = 25
base_pesticides = 10
sens_year = 2025
sens_country = "India"
sens_crop = "Maize"

print(f"\nAnalyzing impact on {sens_crop} in {sens_country}...")

# Calculate sensitivity ranges
rainfall_range = np.linspace(0, 3000, 20)
temp_range = np.linspace(5, 45, 20)
pesticide_range = np.linspace(0, 100, 20)

# Rainfall sensitivity
rainfall_yields = [
    predict_yield(r, base_temp, base_pesticides, sens_year, sens_country, sens_crop)
    for r in rainfall_range
]

# Temperature sensitivity
temp_yields = [
    predict_yield(base_rainfall, t, base_pesticides, sens_year, sens_country, sens_crop)
    for t in temp_range
]
# Pesticide sensitivity
pesticide_yields = [
    predict_yield(base_rainfall, base_temp, p, sens_year, sens_country, sens_crop)
    for p in pesticide_range
]

print("\nSensitivity Results:")
print(f"  Rainfall Impact: {min(rainfall_yields):.0f} - {max(rainfall_yields):.0f} hg/ha")
print(f"  Temperature Impact: {min(temp_yields):.0f} - {max(temp_yields):.0f} hg/ha")
print(f"  Pesticide Impact: {min(pesticide_yields):.0f} - {max(pesticide_yields):.0f} hg/ha")

In [ ]:
# Plot sensitivity analysis
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=rainfall_range, y=rainfall_yields,
    mode='lines+markers', name='Rainfall Effect',
    line=dict(width=3, color='blue')
))

fig.update_layout(
    title='Rainfall Impact on Yield',
    xaxis_title='Rainfall (mm/year)',
    yaxis_title='Yield (hg/ha)',
    hovermode='x unified',
    height=500
)

fig.show()

# Temperature sensitivity plot
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=temp_range, y=temp_yields,
    mode='lines+markers', name='Temperature Effect',
    line=dict(width=3, color='red')
))

fig2.update_layout(
    title='Temperature Impact on Yield',
    xaxis_title='Temperature (°C)',
    yaxis_title='Yield (hg/ha)',
    hovermode='x unified',
    height=500
)

fig2.show()

# Pesticide sensitivity plot
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=pesticide_range, y=pesticide_yields,
    mode='lines+markers', name='Pesticide Effect',
    line=dict(width=3, color='green')
))

fig3.update_layout(
    title='Pesticide Impact on Yield',
    xaxis_title='Pesticides (tonnes)',
    yaxis_title='Yield (hg/ha)',
    hovermode='x unified',
    height=500
)

fig3.show()

# SECTION 3: Batch Predictions from CSV

In [ ]:
# Create sample batch data
batch_data = {
    'Rainfall': [800, 1200, 1500, 900, 1100],
    'Temperature': [20, 25, 28, 22, 26],
    'Pesticides': [5, 10, 15, 8, 12],
    'Area': ['India', 'Brazil', 'Egypt', 'India', 'Brazil'],
    'Item': ['Wheat', 'Maize', 'Rice, paddy', 'Wheat', 'Maize']
}

df_batch = pd.DataFrame(batch_data)
batch_year = 2025

print("=" * 60)
print("BATCH PREDICTIONS")
print("=" * 60)
print("\nInput Data:")
print(df_batch)

# Make batch predictions
print("\nProcessing predictions...")
df_batch['Predicted_Yield_hg_ha'] = df_batch.apply(
    lambda row: predict_yield(
        row['Rainfall'],
        row['Temperature'],
        row['Pesticides'],
        batch_year,
        row['Area'],
        row['Item']
    ),
    axis=1
)

# Convert to other units
df_batch['Predicted_Yield_lbs_acre'] = df_batch['Predicted_Yield_hg_ha'].apply(hg_per_hectare_to_lbs_per_acre)
df_batch['Predicted_Yield_tons_acre'] = df_batch['Predicted_Yield_hg_ha'].apply(hg_per_hectare_to_tons_per_acre)

print("\nResults:")
print(df_batch.to_string())

# Summary statistics
print(f"\nSummary Statistics (hg/ha):")
print(f"  Average Yield: {df_batch['Predicted_Yield_hg_ha'].mean():.0f}")
print(f"  Min Yield: {df_batch['Predicted_Yield_hg_ha'].min():.0f}")
print(f"  Max Yield: {df_batch['Predicted_Yield_hg_ha'].max():.0f}")
print(f"  Std Dev: {df_batch['Predicted_Yield_hg_ha'].std():.0f}")

# SECTION 4: Multi-Scenario Trends & Comparison

In [ ]:
print("\n" + "=" * 60)
print("MULTI-SCENARIO TRENDS & COMPARISON")
print("=" * 60)

# Scenario 1: Yield trends over years
years = list(range(2020, 2031))
wheat_yields = [
    predict_yield(1200, 25, 10, year, "India", "Wheat")
    for year in years
]
maize_yields = [
    predict_yield(1200, 25, 10, year, "Brazil", "Maize")
    for year in years
]

print("\nYear-over-Year Trend (2020-2030):")
print(f"  Wheat (India): {wheat_yields[0]:.0f} -> {wheat_yields[-1]:.0f} hg/ha")
print(f"  Maize (Brazil): {maize_yields[0]:.0f} -> {maize_yields[-1]:.0f} hg/ha")

# Plot trend
fig_trend = go.Figure()

fig_trend.add_trace(go.Scatter(
    x=years, y=wheat_yields,
    mode='lines+markers', name='Wheat (India)',
    line=dict(width=3, color='gold')
))

fig_trend.add_trace(go.Scatter(
    x=years, y=maize_yields,
    mode='lines+markers', name='Maize (Brazil)',
    line=dict(width=3, color='yellow')
))

fig_trend.update_layout(
    title='Predicted Yield Trends (2020-2030)',
    xaxis_title='Year',
    yaxis_title='Yield (hg/ha)',
    hovermode='x unified',
    height=500
)

fig_trend.show()

In [ ]:
# Scenario 2: Country comparison
countries = ["India", "Brazil", "Egypt", "USA", "China"]
wheat_by_country = [
    predict_yield(1200, 25, 10, 2025, country, "Wheat")
    for country in countries
]

fig_country = go.Figure(data=[
    go.Bar(
        x=countries,
        y=wheat_by_country,
        marker_color='lightblue',
        text=[f'{y:.0f}' for y in wheat_by_country],
        textposition='auto'
    )
])

fig_country.update_layout(
    title='Wheat Yield Comparison Across Countries (2025)',
    xaxis_title='Country',
    yaxis_title='Yield (hg/ha)',
    height=500
)

fig_country.show()

# SECTION 5: Image-Based Crop Analysis

In [ ]:
print("\n" + "=" * 60)
print("IMAGE-BASED CROP ANALYSIS")
print("=" * 60)

# Sample image analysis function (simulated)
def analyze_crop_image(image_url):
    """Analyze crop image and extract visual indicators."""
    try:
        response = requests.get(image_url, timeout=5)
        img = Image.open(BytesIO(response.content))
        
        # Simulated analysis
        analysis = {
            'Crop Type': 'Wheat',
            'Health Status': 'Good',
            'Estimated Green Coverage': '85%',
            'Visible Issues': 'None detected',
            'Irrigation Status': 'Adequate',
            'Soil Quality': 'Good',
            'Estimated Rainfall': '1200 mm/year'
        }
        
        return img, analysis
    except Exception as e:
        print(f"Error loading image: {e}")
        return None, None

# Use a sample crop image
sample_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5e/Wheat_grain.jpg/800px-Wheat_grain.jpg"

print(f"\nLoading sample image...")
img, analysis = analyze_crop_image(sample_image_url)

if img and analysis:
    print("\nImage Analysis Results:")
    for key, value in analysis.items():
        print(f"  {key}: {value}")
    
    # Display image
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Crop Sample Image')
    plt.tight_layout()
    plt.show()
else:
    print("\nImage analysis not available in this environment.")

# SECTION 6: Advanced Crop Advisor Bot

In [ ]:
print("\n" + "=" * 60)
print("CROP ADVISOR BOT")
print("=" * 60)

def analyze_symptoms(text):
    """Analyze symptoms and return detected issue type."""
    text_lower = text.lower()
    detected_issues = {}
    
    for keyword, issue in issue_keywords.items():
        if keyword in text_lower:
            detected_issues[issue] = detected_issues.get(issue, 0) + 1
    
    if detected_issues:
        return max(detected_issues.items(), key=lambda x: x[1])[0]
    return "General Concern"

# Example 1: Wheat disease analysis
print("\nExample 1: Wheat with Yellow Leaves")
print("-" * 40)
crop_choice = "Wheat"
symptoms = "Yellow leaves with brown spots on the lower canopy"

detected_issue = analyze_symptoms(symptoms)
advice = crop_advice.get(crop_choice, {}).get(detected_issue, "Consult local extension services.")

print(f"Crop: {crop_choice}")
print(f"Symptoms: {symptoms}")
print(f"Detected Issue: {detected_issue}")
print(f"Recommendation: {advice}")

# Example 2: Maize water stress
print("\nExample 2: Maize with Water Stress")
print("-" * 40)
crop_choice = "Maize"
symptoms = "Wilting and dry soil, especially during tasseling stage"

detected_issue = analyze_symptoms(symptoms)
advice = crop_advice.get(crop_choice, {}).get(detected_issue, "Consult local extension services.")

print(f"Crop: {crop_choice}")
print(f"Symptoms: {symptoms}")
print(f"Detected Issue: {detected_issue}")
print(f"Recommendation: {advice}")

# Example 3: General pest management
print("\nExample 3: Rice with Pest Damage")
print("-" * 40)
crop_choice = "Rice, paddy"
symptoms = "Pest infestations and disease spots on leaves"

detected_issue = analyze_symptoms(symptoms)
advice = crop_advice.get(crop_choice, {}).get(detected_issue, "Consult local extension services.")

print(f"Crop: {crop_choice}")
print(f"Symptoms: {symptoms}")
print(f"Detected Issue: {detected_issue}")
print(f"Recommendation: {advice}")

# SECTION 7: Custom Prediction Interface

In [ ]:
# Create custom prediction scenarios
print("\n" + "=" * 60)
print("CUSTOM PREDICTION SCENARIOS")
print("=" * 60)

# Scenario definitions
scenarios = [
    {
        'name': 'Optimal Conditions',
        'rainfall': 1500,
        'temperature': 25,
        'pesticides': 15,
        'year': 2025,
        'country': 'India',
        'crop': 'Wheat'
    },
    {
        'name': 'Drought Conditions',
        'rainfall': 600,
        'temperature': 30,
        'pesticides': 8,
        'year': 2025,
        'country': 'India',
        'crop': 'Wheat'
    },
    {
        'name': 'Low Input',
        'rainfall': 1000,
        'temperature': 22,
        'pesticides': 3,
        'year': 2025,
        'country': 'India',
        'crop': 'Wheat'
    }
]

results = []
for scenario in scenarios:
    yield_val = predict_yield(
        scenario['rainfall'],
        scenario['temperature'],
        scenario['pesticides'],
        scenario['year'],
        scenario['country'],
        scenario['crop']
    )
    
    results.append({
        'Scenario': scenario['name'],
        'Rainfall (mm)': scenario['rainfall'],
        'Temp (°C)': scenario['temperature'],
        'Pesticides (t)': scenario['pesticides'],
        'Yield (hg/ha)': yield_val if yield_val else 0
    })

results_df = pd.DataFrame(results)
print("\nScenario Comparison:")
print(results_df.to_string(index=False))

# Visualize scenarios
fig_scenario = go.Figure(data=[
    go.Bar(
        x=results_df['Scenario'],
        y=results_df['Yield (hg/ha)'],
        marker_color=['green', 'red', 'orange'],
        text=[f'{y:.0f}' for y in results_df['Yield (hg/ha)']],
        textposition='auto'
    )
])

fig_scenario.update_layout(
    title='Wheat Yield Under Different Scenarios',
    xaxis_title='Scenario',
    yaxis_title='Yield (hg/ha)',
    height=500
)

fig_scenario.show()

# SECTION 8: Export Results

In [ ]:
print("\n" + "=" * 60)
print("EXPORT RESULTS")
print("=" * 60)

# Export batch predictions to CSV
csv_path = '/content/crop_yield_predictions.csv'
df_batch[['Area', 'Item', 'Rainfall', 'Temperature', 'Pesticides', 'Predicted_Yield_hg_ha', 'Predicted_Yield_lbs_acre']].to_csv(csv_path, index=False)
print(f"\nBatch predictions saved to: {csv_path}")
print("Available columns:")
print("  - Area, Item, Rainfall, Temperature, Pesticides")
print("  - Predicted_Yield_hg_ha, Predicted_Yield_lbs_acre")

# Summary statistics
print("\nExport Summary:")
print(f"  Total predictions: {len(df_batch)}")
print(f"  Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Model: RandomForest Regressor")
print(f"  Features: {len(all_features)}")

# Summary

This notebook provides comprehensive crop yield prediction capabilities:

1. **Single Predictions** - Quick yield estimates for specific scenarios
2. **Sensitivity Analysis** - Understand parameter impacts on yield
3. **Batch Processing** - Multiple predictions from CSV data
4. **Trend Analysis** - Year-over-year and cross-country comparisons
5. **Image Analysis** - Visual crop health assessment
6. **Crop Advisor** - AI-powered recommendations for crop issues
7. **Export** - Save results to CSV format

**Key Features:**
- Support for 113 countries and 9+ crop types
- Unit conversion (hg/ha, lbs/acre, tons/acre)
- Interactive visualizations with Plotly
- Comprehensive crop disease knowledge base
- Scenario-based comparative analysis

**Next Steps:**
- Modify parameters in each section to suit your needs
- Upload your own CSV data for batch predictions
- Use the advisor bot for specific crop issues
- Compare different farming scenarios